# 03b: APTOS external inference, all seeds (v2)

Applies every EyePACS-trained v2 model (11 architectures x 3 seeds = 33), **unchanged**, to
the full APTOS 2019 set (3,662 images). No training, tuning or selection happens here.

- Resumable: skips any model whose APTOS prediction file already exists. The nine seed-0
  non-ViT files from notebook 03 are kept, so this runs the remaining 24.
- Strict checkpoint loading (`strict=True`) into weight-free skeletons: any mismatch fails
  loudly instead of loading partially.
- Same processed APTOS PNGs and the same eval transform as training.
- Output: `aptos_predictions_external_<model>[_seedK]_v2.csv` (float64) in `03_predictions`,
  and a merged summary of all 33 in `99_audit/aptos_inference_summary_v2.json`.

APTOS has no patient identifiers, so each image is its own unit.

A T4 is plenty. About 30-40 minutes, mostly checkpoint downloads (RETFound is 1.2 GB each).
Needs the `HF_TOKEN` secret only to fetch the tiny RETFound config. Run cells 1-4.

In [3]:
# Cell 1: auth, deps, Drive helpers
!pip -q install timm transformers
import io, os, json, time, datetime, threading
import numpy as np, pandas as pd, torch
import torch.nn as nn
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import auth, userdata
auth.authenticate_user()
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')

F_CHECKPOINTS = '1DAAXJ2fjk2zAzx7PpGaWsk4BXudeA5Le'
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'
F_AUDIT       = '1zuvfcxC3umHAq_uf7Vx9omMEgUEuQhTt'
APTOS_LABELS  = '18oFRy0GwLjOooXvCHDFSRxoEe9q7JR61'
APTOS_IMGDIR  = '1dVSFb5qx41ldizm-4UvTaZpSuXIRmolZ'
RETFOUND_REPO = 'iszt/RETFound_mae_meh'
NUM_WORKERS   = min(8, os.cpu_count())

MODELS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
REPLICATES = [0, 1, 2]
def tag(k): return '' if k == 0 else f'_seed{k}'

def dl(fid, path, service=None):
    s = service or svc
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, s.files().get_media(fileId=fid), chunksize=64*1024*1024); done = False
        while not done: _, done = d.next_chunk()

def find_id(name, folder):
    r = svc.files().list(q=f"'{folder}' in parents and name='{name}' and trashed=false",
                         fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None

def up(path, folder, mime='application/octet-stream', overwrite=False):
    media = MediaFileUpload(path, mimetype=mime, resumable=True); name = os.path.basename(path)
    if overwrite:
        fid = find_id(name, folder)
        if fid: svc.files().update(fileId=fid, media_body=media).execute(); print('  updated', name); return
    svc.files().create(body={'name': name, 'parents': [folder]}, media_body=media, fields='id').execute()
    print('  uploaded', name)

def list_folder(folder):
    out, tok = [], None
    while True:
        r = svc.files().list(q=f"'{folder}' in parents and trashed=false",
                             fields='files(id,name),nextPageToken', pageToken=tok, pageSize=1000).execute()
        out += r.get('files', []); tok = r.get('nextPageToken')
        if not tok: break
    return out

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE', '| workers:', NUM_WORKERS)

GPU: NVIDIA A100-SXM4-40GB | workers: 8


In [4]:
# Cell 2: APTOS labels and images (parallel download), verify full coverage
dl(APTOS_LABELS, 'aptos_labels.csv')
aptos = pd.read_csv('aptos_labels.csv'); aptos['image_id'] = aptos['image_id'].astype(str)
print('APTOS:', len(aptos), 'images | referable rate', round(aptos['binary_label'].mean(), 4))

IMG = '/content/aptos'; os.makedirs(IMG, exist_ok=True)
files = [f for f in list_folder(APTOS_IMGDIR) if f['name'].lower().endswith('.png')]
_local = threading.local()
def _svc():
    if not hasattr(_local, 's'): _local.s = gbuild('drive', 'v3')
    return _local.s
def _get(f):
    p = os.path.join(IMG, f['name'])
    if os.path.exists(p) and os.path.getsize(p) > 0: return
    for a in range(4):
        try: dl(f['id'], p, _svc()); return
        except Exception:
            if a == 3: raise
            time.sleep(2 ** a)
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    for i, fu in enumerate(as_completed([ex.submit(_get, f) for f in files]), 1):
        fu.result()
        if i % 1000 == 0: print(f'  {i}/{len(files)} ({time.time()-t0:.0f}s)')
INDEX = {os.path.splitext(n)[0]: os.path.join(IMG, n) for n in os.listdir(IMG)}
missing = sorted(set(aptos['image_id']) - set(INDEX))
print('images on disk:', len(INDEX), '| missing:', len(missing))
if missing: raise SystemExit('APTOS images missing. Stop and report.')

APTOS: 3662 images | referable rate 0.4061


  1000/3662 (99s)
  2000/3662 (197s)
  3000/3662 (294s)
images on disk: 3662 | missing: 0


In [5]:
# Cell 3: weight-free skeletons, strict checkpoint loading, predict
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score
from transformers import AutoConfig, AutoModel
from PIL import Image

MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
TF_EVAL = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i): return TF_EVAL(Image.open(INDEX[self.df.at[i, 'image_id']]).convert('RGB')), i

class RETFound(nn.Module):
    def __init__(self, nc):
        super().__init__()
        cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
    def forward(self, x):
        h = self.base(pixel_values=x).last_hidden_state
        return self.classifier(self.fc_norm(h[:, 1:].mean(dim=1)))

def load_model(name, k, dev):
    ck = f'eyepacs_{name}{tag(k)}_v2.pt'
    fid = find_id(ck, F_CHECKPOINTS)
    if not fid: raise FileNotFoundError(f'{ck} not in 02_checkpoints')
    dl(fid, ck)
    blob = torch.load(ck, map_location='cpu', weights_only=False); cfg = blob['config']
    m = RETFound(cfg['nc']) if cfg['arch'] == 'retfound' else \
        timm.create_model(cfg['arch'], pretrained=False, num_classes=cfg['nc'])
    m.load_state_dict(blob['state_dict'], strict=True)
    os.remove(ck)                                   # free local disk
    return m.to(dev).eval(), cfg, blob

@torch.no_grad()
def predict(model, dev, bs):
    dl_ = DataLoader(ImgDS(aptos), batch_size=bs, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    probs = np.zeros(len(aptos), dtype=np.float64)
    for x, idx in dl_:
        out = model(x.to(dev, non_blocking=True)).float()
        p = torch.sigmoid(out[:, 0]) if out.shape[1] == 1 else torch.softmax(out, 1)[:, 2:].sum(1)
        probs[idx.numpy()] = p.double().cpu().numpy()
    return probs
print('ready')

ready


In [6]:
# Cell 4: run every missing (replicate, model); merge summary of all 33
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
have = {f['name'] for f in list_folder(F_PREDICTIONS)}
todo = [(k, m) for k in REPLICATES for m in MODELS
        if f'aptos_predictions_external_{m}{tag(k)}_v2.csv' not in have]
print(f'to run: {len(todo)} of {len(REPLICATES) * len(MODELS)}')

# carry forward the existing summary so the merged file covers all 33
summary = {}
sid = find_id('aptos_inference_summary_v2.json', F_AUDIT)
if sid:
    dl(sid, 'prev_summary.json')
    for r in json.load(open('prev_summary.json')).get('models', []):
        summary[r.get('label', r['model'])] = {**r, 'label': r.get('label', r['model']), 'replicate': r.get('replicate', 0)}

for i, (k, name) in enumerate(todo, 1):
    label = name + (tag(k) or '')
    print(f'\n=== [{i}/{len(todo)}] {label} ===')
    t0 = time.time()
    try:
        model, cfg, blob = load_model(name, k, dev)
    except Exception as e:
        print('  FAILED to load:', e); continue
    pr = predict(model, dev, 32 if cfg['arch'] == 'retfound' else 128)
    auc = float(roc_auc_score(aptos['binary_label'].values, pr))
    out = f'aptos_predictions_external_{name}{tag(k)}_v2.csv'
    pd.DataFrame({'image_id': aptos['image_id'], 'patient_id': aptos['image_id'], 'eye': 'unknown',
                  'dataset': 'aptos', 'split': 'external',
                  'true_grade': aptos['level'].astype(int), 'true_label': aptos['binary_label'].astype(int),
                  'predicted_probability': pr, 'predicted_class': (pr >= 0.5).astype(int),
                  'model_name': name, 'replicate': k}).to_csv(out, index=False, float_format='%.17g')
    up(out, F_PREDICTIONS, 'text/csv')
    summary[label] = {'label': label, 'model': name, 'replicate': k, 'aptos_auc': round(auc, 4),
                      'eyepacs_val_auc': round(float(blob.get('best_val_auc', float('nan'))), 4)}
    print(f'  APTOS AUC {auc:.4f}  (EyePACS val {blob.get("best_val_auc", float("nan")):.4f})  {time.time()-t0:.0f}s')
    del model; torch.cuda.empty_cache()

rows = sorted(summary.values(), key=lambda r: (r['model'], r['replicate']))
rec = {'generated': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'n_images': int(len(aptos)), 'referable_rate': round(float(aptos['binary_label'].mean()), 4),
       'grade_dist': {int(a): int(b) for a, b in aptos['level'].value_counts().sort_index().items()},
       'models': rows, 'note': 'AUC is descriptive only. No selection is performed on APTOS.'}
with open('aptos_inference_summary_v2.json', 'w') as f: json.dump(rec, f, indent=2)
up('aptos_inference_summary_v2.json', F_AUDIT, 'application/json', overwrite=True)

done = {f['name'] for f in list_folder(F_PREDICTIONS)}
got = sum(f'aptos_predictions_external_{m}{tag(k)}_v2.csv' in done for k in REPLICATES for m in MODELS)
print(f'\nAPTOS prediction files: {got}/33')
print(pd.DataFrame(rows)[['label','aptos_auc','eyepacs_val_auc']].to_string(index=False))

to run: 24 of 33

=== [1/24] vit_base_binary ===
  uploaded aptos_predictions_external_vit_base_binary_v2.csv
  APTOS AUC 0.9626  (EyePACS val 0.8683)  40s

=== [2/24] vit_base_5class ===
  uploaded aptos_predictions_external_vit_base_5class_v2.csv
  APTOS AUC 0.9679  (EyePACS val 0.8632)  40s

=== [3/24] densenet121_binary_seed1 ===
  uploaded aptos_predictions_external_densenet121_binary_seed1_v2.csv
  APTOS AUC 0.9655  (EyePACS val 0.8801)  18s

=== [4/24] densenet121_5class_seed1 ===
  uploaded aptos_predictions_external_densenet121_5class_seed1_v2.csv
  APTOS AUC 0.9555  (EyePACS val 0.8745)  19s

=== [5/24] efficientnet_b3_binary_seed1 ===
  uploaded aptos_predictions_external_efficientnet_b3_binary_seed1_v2.csv
  APTOS AUC 0.9541  (EyePACS val 0.8527)  22s

=== [6/24] efficientnet_b3_5class_seed1 ===
  uploaded aptos_predictions_external_efficientnet_b3_5class_seed1_v2.csv
  APTOS AUC 0.9430  (EyePACS val 0.8610)  18s

=== [7/24] resnet50_binary_seed1 ===
  uploaded aptos_predic

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

  uploaded aptos_predictions_external_retfound_binary_seed1_v2.csv
  APTOS AUC 0.9598  (EyePACS val 0.8788)  88s

=== [12/24] retfound_5class_seed1 ===
  uploaded aptos_predictions_external_retfound_5class_seed1_v2.csv
  APTOS AUC 0.9607  (EyePACS val 0.8778)  104s

=== [13/24] retfound_adversarial_seed1 ===
  uploaded aptos_predictions_external_retfound_adversarial_seed1_v2.csv
  APTOS AUC 0.9583  (EyePACS val 0.8793)  76s

=== [14/24] densenet121_binary_seed2 ===
  uploaded aptos_predictions_external_densenet121_binary_seed2_v2.csv
  APTOS AUC 0.9477  (EyePACS val 0.8796)  18s

=== [15/24] densenet121_5class_seed2 ===
  uploaded aptos_predictions_external_densenet121_5class_seed2_v2.csv
  APTOS AUC 0.9637  (EyePACS val 0.8826)  17s

=== [16/24] efficientnet_b3_binary_seed2 ===
  uploaded aptos_predictions_external_efficientnet_b3_binary_seed2_v2.csv
  APTOS AUC 0.9507  (EyePACS val 0.8554)  19s

=== [17/24] efficientnet_b3_5class_seed2 ===
  uploaded aptos_predictions_external_effici